## PROJECT-4. Решение комплексной бизнес-задачи - увеличение среднего чека

#### ЗАДАЧИ (ФОРМАЛИЗОВАННЫЕ)

1. Построить рекомендательную систему, благодаря которой можно будет предлагать клиентам интересные им курсы. Для этого потребуется подготовить и проанализировать имеющиеся данные.
2. Составить итоговую таблицу с рекомендациями, снабдив её необходимыми комментариями, и представить  продакт-менеджеру.
3. Проанализировать результаты А/Б-теста, проведённого после внедрения фичи, сделать вывод.

## 1.

### Подготовка данных с помощью SQL

#### Код в Metabase >>>

Используем данный код в 
##### Metabase 
для получения файла, который скачаем в 
##### csv-формате 

01. | with users_with_courses as (
02. | select distinct user_id,
03. |     resource_id
04. | from final.carts as c
05. |     join final.cart_items ci on c.id = ci.cart_id
06. | where ci.resource_type = 'Course' and c.state = 'successful'
07. | order by 1),
08. | counter as (
09. | select user_id,
10. |     count(resource_id) as cc
11. | from users_with_courses
12. | group by user_id)
13. | select c.user_id, u.resource_id
14. | from counter as c
15. |     join users_with_courses as u on c.user_id = u.user_id
16. | where cc > 1
17. | order by 1, 2

##### >>> file 'result_carts'

### ОБРАБОТКА ДАННЫХ

In [1]:
# Импортируем необходимые библиотеки и модули:
import pandas as pd
import numpy as np
import itertools 
from scipy.stats import norm 
import math

In [2]:
# Открываем  файл, скаченный из Metabase после обработки в SQL:
df = pd.read_csv('data/result_carts.csv')
df

,user_id,resource_id
0,51,516
1,51,1099
2,6117,356
3,6117,357
4,6117,1125
...,...,...
34069,2188926,515
34070,2188926,743
34071,2190141,756
34072,2190141,794


In [3]:
df = df.rename(columns = {'user_id': 'Users', 'resource_id': 'Courses'}) # Переименовываем столбцы

df_users = df.groupby('Users')['Courses'].apply(lambda x: list(np.unique(x))).reset_index() # Группируем по пользователям с удалением дубликатов курсов
df_users

,Users,Courses
0,51,"[516, 1099]"
1,6117,"[356, 357, 1125]"
2,10275,"[553, 1147]"
3,10457,"[361, 1138]"
4,17166,"[356, 357]"
...,...,...
12651,2179430,"[566, 750]"
12652,2186581,"[794, 864, 1129]"
12653,2187601,"[356, 553, 571, 765, 912]"
12654,2188926,"[515, 743]"


In [4]:
# Группируем пары курсов и записываем данные в отдельную переменную с помощью метода combinations модуля itertools
courses_pairs = []
for course in df_users['Courses']:
    for pair in itertools.combinations(list(course), 2):
        courses_pairs.append(pair)

In [5]:
courses_pairs

[(np.int64(516), np.int64(1099)),
 (np.int64(356), np.int64(357)),
 (np.int64(356), np.int64(1125)),
 (np.int64(357), np.int64(1125)),
 (np.int64(553), np.int64(1147)),
 (np.int64(361), np.int64(1138)),
 (np.int64(356), np.int64(357)),
 (np.int64(1125), np.int64(1140)),
 (np.int64(551), np.int64(745)),
 (np.int64(553), np.int64(745)),
 (np.int64(551), np.int64(1138)),
 (np.int64(553), np.int64(568)),
 (np.int64(514), np.int64(517)),
 (np.int64(514), np.int64(566)),
 (np.int64(517), np.int64(566)),
 (np.int64(363), np.int64(511)),
 (np.int64(363), np.int64(562)),
 (np.int64(363), np.int64(563)),
 (np.int64(511), np.int64(562)),
 (np.int64(511), np.int64(563)),
 (np.int64(562), np.int64(563)),
 (np.int64(568), np.int64(745)),
 (np.int64(509), np.int64(553)),
 (np.int64(509), np.int64(745)),
 (np.int64(553), np.int64(745)),
 (np.int64(1125), np.int64(1144)),
 (np.int64(509), np.int64(568)),
 (np.int64(509), np.int64(672)),
 (np.int64(568), np.int64(672)),
 (np.int64(516), np.int64(552)),


In [6]:
# Исключаем дубликаты, смотрим количество различных уникальных пар курсов, которые встречаются вместе в покупках пользователей:
len(set(courses_pairs))

3989

#### Определяем самую популярную пару курсов

In [7]:
# Для этого создадим отдельный датафрейм:
df_pairs = pd.DataFrame(courses_pairs, columns = ['Course1', 'Course2'])
df_pairs

,Course1,Course2
0,516,1099
1,356,357
2,356,1125
3,357,1125
4,553,1147
...,...,...
40012,765,912
40013,515,743
40014,756,794
40015,756,1185


In [8]:
# Добавляем столбец 'pairs', где выводим приведённые пары курсов вместе:
df_pairs['pairs'] = df_pairs['Course1'].astype(str) + ',' + df_pairs['Course2'].astype(str)
df_pairs

,Course1,Course2,pairs
0,516,1099,"516,1099"
1,356,357,"356,357"
2,356,1125,"356,1125"
3,357,1125,"357,1125"
4,553,1147,"553,1147"
...,...,...,...
40012,765,912,"765,912"
40013,515,743,"515,743"
40014,756,794,"756,794"
40015,756,1185,"756,1185"


In [9]:
# Сгруппируем датафрейм по уникальным парам курсов, приобретённым каждым клиентом и посчитаем количество таких покупок:
df_grouped = df_pairs.groupby('pairs').count().reset_index()
df_grouped = df_grouped[['pairs', 'Course1']].rename(columns = {'Course1': 'pairs_count'})
df_grouped['pairs_list'] = df_grouped['pairs'].str.split(',')

# Произведём сортировку в датафрейме по количеству от наиболее покупаемых - популярных пар курсов к менее покупаемым:
df_grouped = df_grouped.sort_values('pairs_count', ascending = False)
df_grouped

,pairs,pairs_count,pairs_list
2066,"551,566",797,"[551, 566]"
1624,"515,551",417,"[515, 551]"
911,"489,551",311,"[489, 551]"
1973,"523,551",304,"[523, 551]"
2462,"566,794",290,"[566, 794]"
...,...,...,...
2875,"672,743",1,"[672, 743]"
1883,"519,1184",1,"[519, 1184]"
2873,"672,741",1,"[672, 741]"
2871,"672,1188",1,"[672, 1188]"


Самая популярная пара курсов - это [551, 566] с наибольшим количеством покупок (797), которая расположена в самом верху списка.

### Устанавливаем минимальным порог (90-процентиль) для значений повторных покупок уникальных пар курсов, приобретённых каждым клиентом

In [10]:
# Оставим уникальные пары курсов с наибольшим количеством повторных покупок, а минимальные исключим
# - используем 90-процентиль как минимальный порог, запишем данное условие в переменную "mt" (minimum threshold):
mt = np.percentile(df_grouped['pairs_count'], 90)
display(mt)

np.float64(22.0)

In [11]:
# Создаём новый DataFrame с условием минимального порога для значений повторных покупок - "mt"
df_popular_pairs=df_grouped[df_grouped['pairs_count']>mt]
df_popular_pairs

,pairs,pairs_count,pairs_list
2066,"551,566",797,"[551, 566]"
1624,"515,551",417,"[515, 551]"
911,"489,551",311,"[489, 551]"
1973,"523,551",304,"[523, 551]"
2462,"566,794",290,"[566, 794]"
...,...,...,...
499,"361,672",23,"[361, 672]"
1072,"502,507",23,"[502, 507]"
1900,"519,571",23,"[519, 571]"
862,"368,840",23,"[368, 840]"


Данный DataFrame (df_popular_pairs) с наибольшим количеством повторных покупок будем использовать на этапе оформления таблицы с рекомендациями к основному курсу.

## 2.

### Оформление таблицы с рекомендациями для продакт-менеджера и отдела маркетинга

На данном этапе составляем таблицу с 3-мя столбцами: 
1. курс, к которому идёт рекомендация
2. Курс для рекомендации № 1 (самый популярный).
3. Курс для рекомендации № 2 (второй по популярности).

Для этого , создадим функцию, которая будет рекомендовать два наиболее популярных курса

In [12]:
# Обратимся к первоначальному датафрейму df: выделим курсы, к которым будут идти рекомендации в переменную - main_course
main_course = df['Courses'].unique()
main_course

array([ 516, 1099,  356,  357, 1125,  553, 1147,  361, 1138, 1140,  551,
        745,  568,  514,  517,  566,  363,  511,  562,  563,  509, 1144,
        672,  552,  571,  513, 1141,  744,  862,  679,  750,  800,  569,
        840,  765, 1187, 1100, 1103,  502,  564,  865,  764, 1139, 1186,
        366,  367,  519,  809,  515,  912,  489,  523,  864, 1101, 1146,
        776,  671,  753,  829,  490, 1102,  803,  659,  909,  794,  518,
        907,  777,  908,  360,  813,  835,  741,  752,  814, 1115, 1116,
       1161,  863,  743,  504,  572,  810, 1124, 1128,  742, 1104,  503,
        664,  507,  570, 1185, 1198,  365,  359,  791, 1156,  362, 1184,
        911,  358, 1160,  757,  508, 1181,  755, 1145, 1188,  756,  866,
        749,  368,  364,  834, 1152,  670, 1199,  836, 1201, 1129, 1182,
        902,  837, 1200,  833,  830])

In [13]:
# Создадим функцию recommend, которая будет рекомендовать 1-ый и 2-ой по популярности курс к основному курсу:
def recommend(main_course):
    course_list = []
    for i in df_popular_pairs['pairs_list']:               
        if int(i[0]) == main_course:            
            course_list.append(i)   
    return course_list[:2] # ставим ограничение для вывода 2-х рекомендаций

In [14]:
recommend(516)

[['516', '745'], ['516', '553']]

In [15]:
# Определяем уникальное количество курсов:
unique_courses = df.Courses.unique()
len(unique_courses)

126

In [16]:
# Теперь, создадим цикл, который для каждого основного курса извлекает 2-ое значение пары согласно уникальным значениям курсов:

recommended = pd.DataFrame(columns = ['Рекомендация 1', 'Рекомендация 2'])
for i in unique_courses:
    if len(recommend(i)) == 2: # задаём условие вывода функцией 2-х рекомендаций
        recommended.loc[i] = [int(recommend(i)[0][1]), int(recommend(i)[1][1])] # выводим 1 и 2 рекомендации
    elif len(recommend(i)) == 1: # задаём условие вывода функцией 1-ой рекомендации
        recommended.loc[i] = [int(recommend(i)[0][1]), np.nan]
    else:                        # задаём условие вывода при отсутствии рекомендации
        recommended.loc[i] = [np.nan, np.nan]

In [17]:
recommended.head()

,Рекомендация 1,Рекомендация 2
516,745.0,553.0
1099,NaN,NaN
356,571.0,357.0
357,571.0,1125.0
1125,1186.0,NaN


##### Для курсов с пустыми ячейками (nan) - производим замену: используем функцию random.choice() , которая выдаёт случайный выбор курсов в качестве рекомендации из списка самых популярных курсов.

In [18]:
# Создадим список популярных курсов, каждый из которых имеет более 100 покупок:  
df_curses=df.groupby('Courses').count().sort_values('Users', ascending=False)
df_curses=df_curses[df_curses['Users']>100]

list_courses=df_curses.index.to_list()

In [19]:
# Заменим пустые значения в рекомендациях на другие курсы из списка популярных: 

recommended = pd.DataFrame(columns = ['Рекомендация 1', 'Рекомендация 2'])
for i in unique_courses:
    if len(recommend(i)) == 2: # задаём условие вывода функцией 2-х рекомендаций
        recommended.loc[i] = [recommend(i)[0][1], recommend(i)[1][1]] # выводим 1 и 2 рекомендации
    elif len(recommend(i)) == 1: # задаём условие вывода функцией 1-ой рекомендации
        recommended.loc[i] = [recommend(i)[0][1], np.random.choice(list_courses)]
    else:
        recommended.loc[i] = [np.random.choice(list_courses), np.random.choice(list_courses)]

In [20]:
recommended.info()

<class 'pandas.core.frame.DataFrame'>
Index: 126 entries, 516 to 830
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Рекомендация 1  126 non-null    object
 1   Рекомендация 2  126 non-null    object
dtypes: object(2)
memory usage: 3.0+ KB


In [21]:
# Выводим основной курс и рекомендации 1, 2 к основному курсу:
# Например, посмотрим ТОП-30:

recommended_courses = recommended.reset_index().rename(columns = {'index': 'Основной курс'})
recommended_courses.head(30)

,Основной курс,Рекомендация 1,Рекомендация 2
0,516,745,553
1,1099,908,367
2,356,571,357
3,357,571,1125
4,1125,1186,1125
5,553,745,568
6,1147,357,672
7,361,551,1138
8,1138,777,513
9,1140,490,1103


### В представленной таблице показаны рекомендации к предложениям курсов для покупки, на основе популярности курсов.

## 3.

### На данном этапе, анализируем результаты А/Б-теста после внедрения фичи с предложением о добавлении в корзину 2-го подходящего курса.

После внедрения предложенной системы тестируем гипотезу с помощью А/Б-теста. Для этого запускаем сплит-тест, где все клиенты случайным образом делятся на А-контрольную и Б-тестовую группы. Тестовой группе показываются рекомендации, а контрольной — нет.

3.1
При следующих условиях теста:
- Средняя конверсия в покупку второго курса, до реализации рекомендаций      -    3,2% 
- Ожидаемая средняя конверсия в покупку 2-го курса, после реализации рекомендаций - 4%
- Уровень достоверности  -  95%
- Статистическая мощность - 80%

С помощью калькулятора достоверности А/Б-тестирования (https://www.evanmiller.org/ab-testing/sample-size.html), определяем минимальный размер выборки в каждом варианте для проведения теста: 7866 > округляем до сотых > 7900 - именно столько человек нужно чтобы получить достоверные результаты при сравнении двух вариантов.

3.2
Теперь, оцениваем результаты 3-х недельного сплит-теста:
- в контрольной группе А: 8732 клиента, оформивших заказ; 293 купили больше одного курса
- в тестовой группе Б:    8847 клиента, оформивших заказ; 347 купили больше одного курса

С помощью калькулятора итогов тестирования (https://abtestguide.com/calc/), определяем, что наблюдаемый коэффициент конверсии для варианта Б - 3,92% на 16,89% выше, чем для варианта A - 3,36%; p-значение = 0.0224. С заданной достоверностью 95% и наблюдаемой мощностью 88.69%, мы можем быть уверены, что результат увеличения конверсии является следствием внесенных нами изменений - добавлением фичи, а не результатом случайности.